## Colab Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
    REPO_PATH = "/content/packt-modern-time-series"

    if not os.path.exists(REPO_PATH):
        os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

    os.chdir(f"{REPO_PATH}/student_notebooks")

    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)

    os.system("pip install -q lightgbm mlforecast")

print(f"✓ Setup complete — {os.getcwd()}")

---

# Module 5 — ML Forecasting with LightGBM
**Type:** [Code With Me]  
**Time:** 45 minutes  
**Job:** Prove that a feature-engineered ML pipeline beats the statistical baselines. Quantify the compute cost. Make an honest recommendation about whether the improvement justifies the infrastructure.

---
## 5.1 — Setup
**[Watch Only]**

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.utils import PredictionIntervals

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
    USE_INTERVALS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts
from src.schemas import validate_forecast
from src.workshop_utils import get_micro_subset
from src.forecast_schema import reshape_mlforecast_cv
from src.plotting import plot_forecast_overlay

print("Setup complete.")
print(f"USE_TUNED_PARAMS = {USE_TUNED_PARAMS}")
print(f"USE_INTERVALS    = {USE_INTERVALS}")

---
## 5.2 — Load Panel and Micro Subset
**[Watch Only]**

---

In [ ]:
panel = load_checkpoint("03_validated_panel")
micro, top_series = get_micro_subset(panel, n=MICRO_SUBSET_N)

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 5.3 — Why ML Forecasting?
**[Watch Only]**

**LightGBM via MLForecast is a global model.** It trains on all series simultaneously — each series becomes a set of training rows. The model learns patterns across the full portfolio, not just within a single series.


**The three feature configurations we will build (5.5A–5.5C):**

| Section | Feature type | What it adds |
|---|---|---|
| **5.5A — Base** | Lags (7, 14, 28), rolling mean, day-of-week + month | Demand history and calendar |
| **5.5B — Rich** | Extended lags, rolling stats, Fourier features, seasonal rolling | More complete numeric/date signal |
| **5.5C — Categorical** | M5 hierarchy labels (category, dept, state, store) as static features | Structural context no univariate model can see |



---
## 5.4 — Load LightGBM Parameters
**[Watch Only]**

---

In [ ]:
params_file = "mlforecast_lgb_tuned.json" if USE_TUNED_PARAMS else "mlforecast_lgb_default.json"
params_path = PARAMS_DIR / params_file

with open(params_path) as f:
    lgb_params = json.load(f)

lgb_params["random_state"] = RANDOM_SEED

print(f"Loaded {len(lgb_params)} params from: {params_file}")

---
## 5.5A — Configure MLForecast (Base)
**[Code With Me — 3 lines]**

Fill in `lags`, `lag_transforms`, and `date_features`. Use weekly, biweekly, and monthly lags. Rolling mean on lag 7 with a 28-day window. Day-of-week and month as date features.

Sections 5.5B and 5.5C define richer configs. You will run all three in 5.6 and compare them in 5.8 (micro) and 5.10 (full panel).

**Fill-in focus:** MLForecast turns time series into supervised learning. Lags provide memory, rolling features summarize recent behavior, and date features provide calendar structure.

---

In [ ]:
mlf = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=[__FILL_IN__, __FILL_IN__, __FILL_IN__],
    lag_transforms={
        7: [RollingMean(window_size=__FILL_IN__)],
    },
    date_features=[__FILL_IN__, __FILL_IN__],
)

print("MLForecast configured.")
print(f"  Lags          : {mlf.ts.lags}")
print(f"  Date features : {mlf.ts.date_features}")
print(f"  Model         : {next(iter(mlf.models.values())).__class__.__name__}")

---
### 5.5B — Feature Engineering Reference: Enhanced Numeric and Date Features
**[Watch Only]**

Feature families:
- **More lags** — demand echoes at 1, 2, 3, 4 weeks
- **Rolling statistics** — mean (level), std (volatility), min/max (range), quantiles (distributional shape)
- **Fourier features** — encode weekly and annual seasonality as sin/cos waves. Better than integer `dayofweek` on short or sparse series because the circular structure is baked into the input rather than learned from scratch.
- **Seasonal rolling mean** — average of the same weekday over the past N weeks; directly encodes the weekly profile

The 5.5A config is deliberately minimal. This block shows the full toolkit. 5.5C adds categorical hierarchy on top.

---

In [ ]:
import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import (
    RollingMean, RollingStd, RollingMin, RollingMax,
    RollingQuantile, SeasonalRollingMean,
)

LAGS = [7, 14, 21, 28]

ROLLING_28 = [
    RollingMean(window_size=28),
    RollingStd(window_size=28),
    RollingMin(window_size=28),
    RollingMax(window_size=28),
    RollingQuantile(p=0.25, window_size=28),
    RollingQuantile(p=0.75, window_size=28),
]

SEASONAL_ROLLING = [
    SeasonalRollingMean(season_length=7, window_size=4),
    SeasonalRollingMean(season_length=7, window_size=8),
]

def fourier_sin1_weekly(dates): return np.sin(2 * np.pi * 1 * dates.dayofweek / 7)
def fourier_cos1_weekly(dates): return np.cos(2 * np.pi * 1 * dates.dayofweek / 7)
def fourier_sin2_weekly(dates): return np.sin(2 * np.pi * 2 * dates.dayofweek / 7)
def fourier_cos2_weekly(dates): return np.cos(2 * np.pi * 2 * dates.dayofweek / 7)
def fourier_sin1_annual(dates): return np.sin(2 * np.pi * 1 * dates.dayofyear / 365.25)
def fourier_cos1_annual(dates): return np.cos(2 * np.pi * 1 * dates.dayofyear / 365.25)

DATE_FEATURES = [
    "month",
    fourier_sin1_weekly, fourier_cos1_weekly,
    fourier_sin2_weekly, fourier_cos2_weekly,
    fourier_sin1_annual, fourier_cos1_annual,
]

mlf_rich = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={
        7:  ROLLING_28 + SEASONAL_ROLLING,
        14: [RollingMean(window_size=28)],
    },
    date_features=DATE_FEATURES,
)

print(f"Rich config: {len(LAGS)} lags, {len(ROLLING_28+SEASONAL_ROLLING)} lag-7 transforms, {len(DATE_FEATURES)} date features")

---
### 5.5C — Static Categorical Features: M5 Product Hierarchy
**[Code With Me — 1 line]**

Fill in the names of the static variables.

Every model so far — Naive, SeasonalNaive, AutoETS, Chronos — processes each series in isolation. LightGBM is the first model in our stack that can use structural information that *spans* series.

The M5 unique_id encodes a full product-store hierarchy. We extract it and pass it as **static features** — the same value for every row of a given series. MLForecast passes them to LightGBM as constant columns at every training row.

Key points:
- `static_features` is passed to `cross_validation()` as a keyword argument — not to the constructor
- LightGBM natively handles pandas `category` dtype — no integer encoding needed. Keeping the original labels preserves business meaning; arbitrary `.cat.codes` would impose a false ordinal structure (store code 3 is not "greater than" store code 2)
- In production, static features come from a product master table, not from parsing series IDs

---

In [ ]:
# M5 unique_id format: {CATEGORY}_{DEPT_NUM}_{ITEM_NUM}_{STATE_ID}_{STORE_NUM}
# Example: FOODS_2_360_WI_2 → cat=FOODS, dept=FOODS_2, state=WI, store=WI_2

def add_m5_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    """Extract M5 hierarchy labels as pandas categoricals. LightGBM handles category dtype natively."""
    parts = df["unique_id"].str.split("_")
    df = df.copy()
    df["cat_id"]   = parts.str[0]
    df["dept_id"]  = parts.str[:2].str.join("_")
    df["state_id"] = parts.str[3]
    df["store_id"] = parts.str[3:5].str.join("_")
    for col in ["cat_id", "dept_id", "state_id", "store_id"]:
        df[col] = df[col].astype("category")
    return df

STATIC_FEATURES = [__FILL_IN__, __FILL_IN__, __FILL_IN__, __FILL_IN__]
micro_cat = add_m5_categoricals(micro)

# static_features is passed to cross_validation(), not to the constructor
mlf_cat = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={7: ROLLING_28 + SEASONAL_ROLLING, 14: [RollingMean(window_size=28)]},
    date_features=DATE_FEATURES,
)

print(f"micro_cat shape: {micro_cat.shape} — static cols: {STATIC_FEATURES}")
micro_cat[["unique_id"] + STATIC_FEATURES].drop_duplicates().head()

**Expected output:**
```
MLForecast configured.
  Lags          : [7, 14, 28]
  Date features : ['dayofweek', 'month']
  Model         : LGBMRegressor
```

---
## 5.6 — Run Micro CV for All Three Configurations
**[Watch Only]**

Run all three feature configs on the 50-series micro subset in one block. This confirms the full pipeline works end-to-end before we evaluate on the full panel.

Target runtime: < 2 min on Colab CPU.

---

In [ ]:
%%time
# Run micro CV for all three feature configurations
# Target runtime: < 2 min on Colab CPU

# ── Base config ───────────────────────────────────────────────────────────────
cv_base_micro = mlf.cross_validation(
    df=micro, h=HORIZON, n_windows=N_WINDOWS, step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
)

# ── Rich config ───────────────────────────────────────────────────────────────
cv_rich_micro = mlf_rich.cross_validation(
    df=micro, h=HORIZON, n_windows=N_WINDOWS, step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
)

# ── Categorical config ────────────────────────────────────────────────────────
cv_cat_micro = mlf_cat.cross_validation(
    df=micro_cat, h=HORIZON, n_windows=N_WINDOWS, step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
    static_features=STATIC_FEATURES,
)

print(f"Base  CV: {cv_base_micro.shape[0]:,} rows × {cv_base_micro.shape[1]} columns")
print(f"Rich  CV: {cv_rich_micro.shape[0]:,} rows × {cv_rich_micro.shape[1]} columns")
print(f"Cat   CV: {cv_cat_micro.shape[0]:,} rows × {cv_cat_micro.shape[1]} columns")

---
## 5.7 — Reshape All Three Micro CV Outputs
**[Watch Only]**

Convert each wide MLForecast output to the workshop forecast schema. All three use `model_col="LGBMRegressor"` but get distinct model names.

---

In [ ]:
ml_base_micro = reshape_mlforecast_cv(cv_base_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM")
ml_rich_micro = reshape_mlforecast_cv(cv_rich_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM-Rich")
ml_cat_micro  = reshape_mlforecast_cv(cv_cat_micro,  model_col="LGBMRegressor", stage="ml", model_name="LightGBM-Cat")

for name, df in [
    ("LightGBM",      ml_base_micro),
    ("LightGBM-Rich", ml_rich_micro),
    ("LightGBM-Cat",  ml_cat_micro),
]:
    print(f"  {name:<15}: {df.shape[0]:,} rows | model = '{df['model'].unique()[0]}'")

**Expected output:**
```
  LightGBM       : 4,200 rows | model = 'LightGBM'
  LightGBM-Rich  : 4,200 rows | model = 'LightGBM-Rich'
  LightGBM-Cat   : 4,200 rows | model = 'LightGBM-Cat'
```
Columns: `unique_id`, `ds`, `y`, `model`, `y_hat`, `lo_80`, `hi_80`, `cutoff`, `stage`

---
## 5.8 — Score and Compare All Three on the Micro Subset
**[Watch Only]**

Validate and score all three micro-subset outputs. The numbers here prove the pipeline is wired correctly. With 50 series the ordering is noise — you need the full panel (5.10) for model selection.

---

In [ ]:
def _get_score(scores_df, metric):
    row = scores_df[scores_df["metric"] == metric]
    return row.iloc[0]["score"] if len(row) > 0 else float("nan")

ml_base_micro_val = validate_forecast(ml_base_micro, artifact_name="05_ml_base_micro")
ml_rich_micro_val = validate_forecast(ml_rich_micro, artifact_name="05_ml_rich_micro")
ml_cat_micro_val  = validate_forecast(ml_cat_micro,  artifact_name="05_ml_cat_micro")

subset_micro      = f"micro_{MICRO_SUBSET_N}"
scores_base_micro = score_forecasts(ml_base_micro_val, subset_name=subset_micro)
scores_rich_micro = score_forecasts(ml_rich_micro_val, subset_name=subset_micro)
scores_cat_micro  = score_forecasts(ml_cat_micro_val,  subset_name=subset_micro)

W = 24
print(f"Micro Subset Comparison ({MICRO_SUBSET_N} series, {N_WINDOWS} windows):")
print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
print(f"  {'-'*66}")
for label, scores in [
    ("Base — 5.5A",        scores_base_micro),
    ("Rich — 5.5B",        scores_rich_micro),
    ("Categorical — 5.5C", scores_cat_micro),
]:
    wmape  = _get_score(scores, "wMAPE")
    bias   = _get_score(scores, "Bias")
    iscore = _get_score(scores, "IntervalScore_80")
    print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")
print()
print(f"Key message: {MICRO_SUBSET_N} series confirms the pipeline works end-to-end.")
print("Any ranking here is noise. Full-panel comparison is in 5.10.")

---
## 5.9 — Load Full-Panel Artifacts
**[Watch Only]**

Load all three precomputed full-panel artifacts (1,000 series × 3 windows). These were produced offline by `src/build_offline_artifacts.py` and are the stable basis for model selection.

---

In [ ]:
ml_full_base = load_checkpoint("05_ml_forecasts")
ml_full_rich = load_checkpoint("05_ml_rich_forecasts")
ml_full_cat  = load_checkpoint("05_ml_cat_forecasts")

print(f"Base : {len(ml_full_base):,} rows — model: {ml_full_base['model'].unique()[0]}")
print(f"Rich : {len(ml_full_rich):,} rows — model: {ml_full_rich['model'].unique()[0]}")
print(f"Cat  : {len(ml_full_cat):,} rows — model: {ml_full_cat['model'].unique()[0]}")

---
## 5.10 — Score and Compare All Three on the Full Panel
**[Watch Only]**

Score all three full-panel artifacts. This comparison — on 1,000 series — is the reliable signal that determines which LightGBM variant moves forward.

---

In [ ]:
subset_label = f"workshop_{WORKSHOP_SUBSET_N}"

scores_full_base = score_forecasts(ml_full_base, subset_name=subset_label)
scores_full_rich = score_forecasts(ml_full_rich, subset_name=subset_label)
scores_full_cat  = score_forecasts(ml_full_cat,  subset_name=subset_label)

W = 24
print(f"Full Panel Comparison ({WORKSHOP_SUBSET_N:,} series, {N_WINDOWS} windows — RELIABLE):")
print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
print(f"  {'-'*66}")
for label, scores in [
    ("Base — 5.5A",        scores_full_base),
    ("Rich — 5.5B",        scores_full_rich),
    ("Categorical — 5.5C", scores_full_cat),
]:
    wmape  = _get_score(scores, "wMAPE")
    bias   = _get_score(scores, "Bias")
    iscore = _get_score(scores, "IntervalScore_80")
    print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")
print()
print("This ranking decides which ML variant moves forward to 5.12.")

**Expected output:**
```
Full Panel Comparison (1,000 series, 3 windows — RELIABLE):
  Config                      wMAPE      Bias  IntervalScore_80
  ------------------------------------------------------------------
  Base — 5.5A                0.XXXX   +0.XXXX           XX.XXXX
  Rich — 5.5B                0.XXXX   +0.XXXX           XX.XXXX
  Categorical — 5.5C         0.XXXX   +0.XXXX           XX.XXXX
```
This ranking (not 5.8) determines which variant is selected in 5.12.

---
## 5.11 — Plot: Selected LightGBM vs Best Baseline
**[Watch Only]**

The full-panel comparison selected the winner. Plot it against SeasonalNaive on a representative micro-subset series to give an intuitive view of what the accuracy gain looks like in practice.

---

In [ ]:
example_id = top_series[0]

baseline_full = load_checkpoint("04_baseline_forecasts")
combined = pd.concat([
    ml_full_cat[ml_full_cat["unique_id"] == example_id],
    baseline_full[
        (baseline_full["unique_id"] == example_id) &
        (baseline_full["model"] == "SeasonalNaive")
    ],
], ignore_index=True)

latest_cutoff = combined["cutoff"].max()

plot_forecast_overlay(
    actuals_df=panel,
    forecasts_df=combined[combined["cutoff"] == latest_cutoff],
    unique_id=example_id,
    models=["LightGBM-Cat", "SeasonalNaive"],
    lookback_days=90,
    title=f"LightGBM-Cat vs SeasonalNaive — {example_id}",
)

**Expected output:** Forecast overlay for one series. LightGBM-Cat and SeasonalNaive forecasts overlaid on 90 days of actuals. LightGBM-Cat should track demand level more closely; the 80% interval should be tighter than SeasonalNaive's.

---
## 5.12 — Confirm ML-Stage Leaderboard & Select Artifact
**[Watch Only]**

Build the full ML-stage leaderboard — all three LightGBM variants against the statistical baselines. This is the final decision point before the artifact moves to module 8.

---

In [ ]:
subset_label = f"workshop_{WORKSHOP_SUBSET_N}"

# baseline_full loaded in 5.11; ML scores computed in 5.10
baseline_scores = score_forecasts(baseline_full, subset_name=subset_label)

from src.evaluation import build_leaderboard
leaderboard_5 = build_leaderboard([
    baseline_scores,
    scores_full_base,
    scores_full_rich,
    scores_full_cat,
])

display_cols   = ["model", "wMAPE", "Bias", "IntervalScore_80"]
available_cols = [c for c in display_cols if c in leaderboard_5.columns]
wmape_lb = (
    leaderboard_5[available_cols]
    .dropna(subset=["wMAPE"])
    .sort_values("wMAPE")
    .reset_index(drop=True)
)
print("ML-Stage Leaderboard (sorted by wMAPE):")
display(wmape_lb)

winner = wmape_lb.iloc[0]["model"]
margin = wmape_lb.iloc[1]["wMAPE"] - wmape_lb.iloc[0]["wMAPE"]
print()
print(f"  Winner         : {winner}  (margin over 2nd: {margin:.4f} wMAPE)")
print(f"  Downstream     : 05_ml_forecasts flows to modules 7 and 8")
print(f"  Cat artifact   : 05_ml_cat_forecasts available for take-home analysis")

---
## 5.13 — Save Downstream Artifacts
**[Watch Only]**

Save the micro ML artifacts for optional take-home analysis. The full-panel checkpoints (`05_ml_forecasts`, `05_ml_rich_forecasts`, `05_ml_cat_forecasts`) were precomputed and are already registered.

---

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

micro_artifacts = {
    "05_ml_base_micro_forecasts.parquet": ml_base_micro_val,
    "05_ml_rich_micro_forecasts.parquet": ml_rich_micro_val,
    "05_ml_cat_micro_forecasts.parquet":  ml_cat_micro_val,
}
for fname, df_val in micro_artifacts.items():
    path = ARTIFACT_DIR / fname
    df_val.to_parquet(path, index=False)
    print(f"  ✓ Micro artifact saved : {fname} ({len(df_val):,} rows)")

print()
print("Full-panel artifacts (precomputed checkpoints):")
print(f"  ✓ 05_ml_forecasts.parquet          ({len(ml_full_base):,} rows) — LightGBM base")
print(f"  ✓ 05_ml_rich_forecasts.parquet      ({len(ml_full_rich):,} rows) — LightGBM-Rich")
print(f"  ✓ 05_ml_cat_forecasts.parquet       ({len(ml_full_cat):,} rows)  — LightGBM-Cat")